# Block D live lab — KMZ vs Nagel

**Papers:** Kelly, Malamud & Zhou (2024), *The Virtue of Complexity in Return Prediction*, *JF*. Nagel (2025), *Seemingly Virtuous Complexity in Return Prediction*, NBER WP 34104.

Where the two papers differ, this notebook follows **Nagel** and offers KMZ's choice as a switch. Precompute (`scripts/precompute_kmz.py`) stores the RFF grid, the kernel forecast and the RFF placebo. Steps 4–6 run live.

**Draws:** 20 RFF draws for the $P$ grid; 100 draws for the headline case ($P = 12{,}000$, ridgeless). KMZ and Nagel average 1,000 draws.


## 0. Setup

In [ ]:
KMZ_RETURN_STD = False  # Nagel default. True = KMZ: divide the target by trailing 12-month vol.
RUN_HOMEWORK = False  # T = 60, 120 (Nagel Appendix C.1). Class run skips this.

from pathlib import Path
import sys
import time
import warnings
warnings.filterwarnings("ignore", message=".*encountered in matmul")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True


def find_data_root():
    for base in [Path.cwd(), *Path.cwd().parents]:
        p = base / "data"
        if (p / "goyal_welch" / "goyal_welch_2025.csv").exists() or (p / "README.md").exists():
            return p.resolve()
    return Path.cwd() / "data"


def display_path(p):
    p = Path(p).resolve()
    for base in p.parents:
        if (base / "requirements.txt").is_file():
            try:
                return p.relative_to(base).as_posix()
            except ValueError:
                break
    return p.name


DATA = find_data_root()
CACHE = DATA / "cache"
SCRIPTS = DATA.parent / "scripts"
if str(SCRIPTS) not in sys.path:
    sys.path.insert(0, str(SCRIPTS))

import precompute_kmz as K

print("KMZ_RETURN_STD", KMZ_RETURN_STD, "(Nagel default is False)")
print("RUN_HOMEWORK", RUN_HOMEWORK)
print("cache", display_path(CACHE))
K.assert_no_lookahead()

NW_LAGS = 6  # Newey–West lags. Nagel's published t is plain OLS: IR × √years.


def ols_alpha(y, X):
    """Homoskedastic OLS intercept. Nagel's t ≈ IR × √(n/12)."""
    n, k = X.shape
    xtx = X.T @ X
    beta = np.linalg.solve(xtx, X.T @ y)
    e = y - X @ beta
    s2 = float(np.dot(e, e)) / (n - k)
    se = np.sqrt(s2 * np.diag(np.linalg.inv(xtx)))
    sig = float(e.std(ddof=1))
    ir = np.nan if sig == 0 else (beta[0] / sig) * np.sqrt(12.0)
    return {
        "alpha_m": float(beta[0]),
        "t_ols": float(beta[0] / se[0]),
        "ir": float(ir),
        "n": int(n),
        "t_ir_years": float(ir * np.sqrt(n / 12.0)),
    }


def capm_ols_nw(strat, mkt, mask):
    m = mask & np.isfinite(strat) & np.isfinite(mkt)
    yv = strat[m]
    X = np.column_stack([np.ones(yv.size), mkt[m]])
    o = ols_alpha(yv, X)
    nw = K.capm_stats(strat, mkt, mask, lags=NW_LAGS)
    o["t_nw"] = nw["tstat"]
    o["alpha_ann"] = 12.0 * o["alpha_m"]
    return o


def span_ols_nw(strat, mkt, other, mask):
    m = mask & np.isfinite(strat) & np.isfinite(mkt) & np.isfinite(other)
    yv = strat[m]
    X = np.column_stack([np.ones(yv.size), mkt[m], other[m]])
    o = ols_alpha(yv, X)
    nw = K.spanning_stats(strat, mkt, other, mask, lags=NW_LAGS)
    o["t_nw"] = nw["tstat"]
    return o


## 1. Data and forecast timing

Welch–Goyal monthly 2025 release. Target $R_{t+1}$ = `ret` − `Rfree`. `ret` is CRSP's S&P 500 value-weighted return with dividends; KMZ and Nagel use the CRSP value-weighted index, so expect small differences.

Every forecast of $R_{t+1}$ (RFF, kernel, vol-mom) uses information through month $t$ only. Training pairs are $(x_{t-T}, R_{t-T+1}),\ldots,(x_{t-1}, R_t)$; the test feature is $x_t$. Inflation is lagged one extra month (CPI delay, Welch & Goyal 2008). In Nagel's eq. 14 the weight $12/78$ sits on the most recent realized return $R_t$.


In [ ]:
df = K.load_goyal()
pack = K.build_design(df, return_std=KMZ_RETURN_STD)
ym, y, X, sigma = pack["yyyymm"], pack["y"], pack["X"], pack["sigma"]
print("predictors:", pack["names"])
print("yyyymm range", int(ym[0]), "to", int(ym[-1]), "N", len(ym))
ok = np.isfinite(X).all(axis=1)
print("first month with 36m expanding std on all 15 predictors:", int(ym[ok][0]))
print("T = 12, so first RFF/kernel/vol-mom forecast is dated", int(ym[ok][0 + K.T_DEFAULT]))
print("eval main", K.EVAL_MAIN, "post", K.EVAL_POST)

cache_path = K.cache_path()
if not cache_path.exists():
    raise SystemExit(f"missing {cache_path}; run scripts/precompute_kmz.py")
blob = pd.read_pickle(cache_path)
print(f"precompute runtime {blob['runtime_s']:.1f}s  seed {blob['seed']}")
print(f"draws: grid {blob['n_draws_grid']}, headline {blob['n_draws_head']} (papers use 1,000)")
S = blob["by_std"][KMZ_RETURN_STD]
print("kernel windows that needed tiny ridge (1e-10):", S["kernel_n_ridge"])


## 2. RFF forecasts and the VoC curve

$S_{i,t}=[\sin(\gamma\omega_i'x_t),\cos(\gamma\omega_i'x_t)]$, $\omega_i\sim N(0,I)$, $\gamma=2$. Dual ridge $\hat r_{t+1}=s_t'Z'(ZZ'+zI)^{-1}r$ with rolling $T=12$. Smaller $P$ reuses the first $P/2$ frequencies of the same draw.

The VoC claim: annualized Sharpe of the timing strategy $\hat r_{t+1}\times R_{t+1}$ rises with $P/T$ even when $R^2_\text{OOS}$ is negative.


In [ ]:
P = np.array(S["grid"]["P"], dtype=float)
zlab = S["grid"]["z"]
sharpe = np.asarray(S["grid"]["sharpe_main"])
r2 = np.asarray(S["grid"]["r2_main"])
pt = P / blob["T"]

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for i, lab in enumerate(zlab):
    ax[0].plot(pt, sharpe[i], marker="o", ms=3, label=lab)
    ax[1].plot(pt, r2[i], marker="o", ms=3, label=lab)
ax[0].set_xscale("log")
ax[1].set_xscale("log")
ax[0].set_xlabel("P/T")
ax[1].set_xlabel("P/T")
ax[0].set_ylabel("annualized Sharpe")
ax[1].set_ylabel(r"$R^2_{OOS}$")
ax[0].set_title("VoC: Sharpe vs P/T")
ax[1].set_title(r"VoC: $R^2_{OOS}$ vs P/T")
ax[1].legend(title="z", fontsize=8, loc="best")
fig.tight_layout()
plt.show()

ridgeless = sharpe[-1]
print("ridgeless Sharpe by P/T:")
for p, s in zip(pt, ridgeless):
    print(f"  P/T={p:7.1f}  Sharpe={s:7.3f}")
s = ridgeless[np.isfinite(ridgeless)]
print(f"ridgeless Sharpe at smallest P/T: {s[0]:.3f}; at largest: {s[-1]:.3f}")
print("overall higher at large P/T than at small P/T?", bool(s[-1] > s[0]))
print("strictly increasing across the grid?", bool(np.all(np.diff(s) > 0)))


The dip at P/T = 1 is the interpolation threshold: with P = T the ridgeless fit passes through every training point and its variance peaks. KMZ's theory predicts it.

## 3. Kernel forecast (from cache)

Gaussian kernel $k(x,x')=\exp(-\tfrac{\gamma^2}{2}\lVert x-x'\rVert^2)$ on the standardized predictors, ridgeless over the same 12-month window.


In [ ]:
print(f"corr(RFF, kernel)  1930–2020: {S['corr_main']:.3f}   (Nagel: 0.93)")
print(f"corr(RFF, kernel)  2021–2025: {S['corr_post']:.3f}")
print("kernel tiny-ridge count:", S["kernel_n_ridge"], "(0 means K was never singular)")


## 4. Vol-timed momentum (live)

Nagel eq. 14, raw excess returns:

$$\hat r^{\text{volmom}}_{t+1} = 0.05 \times \hat\sigma^{-2}_{x,t} \times \sum_{k=0}^{11} \frac{12-k}{78}\, R_{t-k}$$

$k=0$ is $R_t$. $\hat\sigma^2_{x,t}$ is the average variance of the 15 standardized predictors in the 12-month training window. The 0.05 only sets scale.


In [ ]:
t_live = time.time()
yhat_vm = K.volmom_forecast(X, y, blob["T"])
pos_vm, strat_vm = K.strategy_from_yhat(yhat_vm, y, False, sigma)
print(f"vol-mom live {time.time()-t_live:.2f}s")
print("weights k=0..11 (should start at 12/78):", np.round([(12 - k) / 78.0 for k in range(12)], 4))
print("max |live − cache| yhat:", np.nanmax(np.abs(yhat_vm - S["volmom"]["yhat"])))


## 5. Alphas and spanning (live)

Strategy return on the market excess return. Information ratio = (monthly alpha / residual s.d.) × √12. Nagel's published t-statistic is **plain OLS**: \(t = \text{IR}\times\sqrt{\text{years}}\). We also report Newey–West t-statistics with **6 lags**. Then RFF on the market plus (a) the kernel strategy and (b) the vol-mom strategy.

Nagel's alphas are annualized percent of an arbitrarily scaled strategy — compare **t-statistics and IRs**, not alpha levels.


In [ ]:
t_span = time.time()
mask_main = K.eval_mask(ym, *K.EVAL_MAIN)
mask_post = K.eval_mask(ym, *K.EVAL_POST)
strats = {
    "RFF": S["rff"]["strat"],
    "kernel": S["kernel"]["strat"],
    "vol-mom": strat_vm,
}
rows = []
for name, st in strats.items():
    for window, mask in (("1930–2020", mask_main), ("2021–2025", mask_post)):
        capm = capm_ols_nw(st, y, mask)
        rows.append({
            "strategy": name,
            "window": window,
            "n": capm["n"],
            "IR": capm["ir"],
            "t_OLS": capm["t_ols"],
            "t_NW6": capm["t_nw"],
        })
capm_tab = pd.DataFrame(rows)
print(capm_tab.to_string(index=False, float_format=lambda v: f"{v:8.3f}"))
print("t_OLS = homoskedastic OLS intercept; t_NW6 = Newey–West, 6 lags. Nagel reports t_OLS.")

print("\nSpanning: RFF on market + other")
span_rows = []
for other_name, other in (("kernel", strats["kernel"]), ("vol-mom", strats["vol-mom"])):
    for window, mask in (("1930–2020", mask_main), ("2021–2025", mask_post)):
        sp = span_ols_nw(strats["RFF"], y, other, mask)
        span_rows.append({
            "other": other_name,
            "window": window,
            "t_OLS": sp["t_ols"],
            "t_NW6": sp["t_nw"],
            "n": sp["n"],
        })
print(pd.DataFrame(span_rows).to_string(index=False, float_format=lambda v: f"{v:8.3f}"))
print(f"spanning live {time.time()-t_span:.2f}s")


## 6. Placebo (Nagel's specification)

$\tilde r_t = r_t + \xi_t - 0.2\,\xi_{t-1} - 0.2\,\xi_{t-2}$, $\xi_t \sim N(0, 0.01)$ (0.01 is the **variance**). The lagged-return predictor, the market factor and the strategy returns are built from $\tilde r$; the other 14 predictors stay real. RFF placebo is precomputed (headline path, 100 draws, plus 20 paths). Kernel placebo runs live.


In [ ]:
t_pl = time.time()
tilde = S["placebo_tilde"]
Xp, yp, y_fit_p, sig_p = K.replace_lagged_return(pack, tilde)
yhat_ker_p, n_ridge_p = K.rolling_kernel_forecast(Xp, y_fit_p, blob["T"])
ker_p = K.summarize_strategy(yhat_ker_p, yp, sig_p, KMZ_RETURN_STD, ym, mkt=yp)
print(f"kernel placebo live {time.time()-t_pl:.2f}s  tiny-ridge windows {n_ridge_p}")

ph = S["placebo_head"]["main"]["capm"]
ph_ols = capm_ols_nw(S["placebo_head"]["strat"], S["placebo_tilde"], mask_main)
print("(We discuss this result in class.)")
print(f"kernel placebo (same path)  IR={ker_p['main']['capm']['ir']:.3f}  t_NW6={ker_p['main']['capm']['tstat']:.2f}")

pt = np.asarray(S["placebo_paths_t"], float)
n_neg = int((pt < 0).sum())
print(f"RFF placebo t-stat over 20 paths ({int(S.get('n_placebo_path_draws', 5))} draws each): mean={pt.mean():.2f}  min={pt.min():.2f}  max={pt.max():.2f}")
print(f"negative t on {n_neg} of {pt.size} paths")


## 7. Ours vs the papers

Headline case: $T=12$, $P=12{,}000$, ridgeless. Report both `KMZ_RETURN_STD` settings and both windows. Nagel's Table I is 1930–2020, unstandardized returns. Comparison t-columns are **OLS** (Nagel: IR × √years). Newey–West (6 lags) is shown beside them.


In [ ]:
PAPER = {
    "IR RFF": "0.255 (Nagel IA); KMZ IR ~0.3",
    "t RFF (OLS)": "2.42 (Nagel); KMZ 2.6–2.9",
    "IR kernel": "0.306 (t=2.90)",
    "IR vol-mom": "0.388 (t=3.68)",
    "RFF | kernel t (OLS)": "−0.12 (alpha −0.001)",
    "RFF | vol-mom t (OLS)": "0.95 (alpha 0.012)",
    "corr RFF, kernel": "0.93",
    # (We discuss this result in class.)
}

rows = []
for std_flag, lab in ((False, "False (Nagel)"), (True, "True (KMZ)")):
    block = blob["by_std"][std_flag]
    ym_b, y_b = block["yyyymm"], block["y"]
    plc_mkt = block["placebo_tilde"]
    for window, key, lo, hi in (
        ("1930–2020", "main", *K.EVAL_MAIN),
        ("2021–2025", "post", *K.EVAL_POST),
    ):
        mask = K.eval_mask(ym_b, lo, hi)
        rff = capm_ols_nw(block["rff"]["strat"], y_b, mask)
        ker = capm_ols_nw(block["kernel"]["strat"], y_b, mask)
        vm = capm_ols_nw(block["volmom"]["strat"], y_b, mask)
        sk = span_ols_nw(block["rff"]["strat"], y_b, block["kernel"]["strat"], mask)
        sv = span_ols_nw(block["rff"]["strat"], y_b, block["volmom"]["strat"], mask)
        plc = capm_ols_nw(block["placebo_head"]["strat"], plc_mkt, mask)
        corr = block["corr_main"] if key == "main" else block["corr_post"]
        rows.append({
            "KMZ_RETURN_STD": lab,
            "window": window,
            "n": rff["n"],
            "IR RFF": rff["ir"],
            "t RFF OLS": rff["t_ols"],
            "t RFF NW6": rff["t_nw"],
            "IR kernel": ker["ir"],
            "t ker OLS": ker["t_ols"],
            "t ker NW6": ker["t_nw"],
            "IR vol-mom": vm["ir"],
            "t vm OLS": vm["t_ols"],
            "t vm NW6": vm["t_nw"],
            "RFF|ker OLS": sk["t_ols"],
            "RFF|ker NW6": sk["t_nw"],
            "RFF|vm OLS": sv["t_ols"],
            "RFF|vm NW6": sv["t_nw"],
            "corr": corr,
            "plc t OLS": plc["t_ols"],
            "plc t NW6": plc["t_nw"],
            "plc IR": plc["ir"],
        })
tab = pd.DataFrame(rows)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 24)
print("Paper targets (Nagel Table I unless noted). Comparison t-columns are OLS; NW is 6 lags.")
for k, v in PAPER.items():
    print(f"  {k:24s}  {v}")
print()
print(tab.to_string(index=False, float_format=lambda v: f"{v:7.3f}"))
print()
print("Switch currently displayed in the figures:", KMZ_RETURN_STD)

# 20-draw vs 100-draw RFF | kernel spanning. 20-draw yhat was not pickled;
# rebuild from cached X with the same seed (first 20 of the 100 headline frequencies).
print("\nRFF | kernel spanning t at P=12,000 ridgeless (1930–2020), by draw count")
print("(20-draw path rebuilt from cached X + seed; 100-draw path from cache.)")
rng_d = np.random.default_rng(blob["seed"])
omegas_all = K.draw_omegas(rng_d, blob["n_draws_head"], blob["P_head"] // 2, X.shape[1])
omegas20 = omegas_all[: blob["n_draws_grid"]]
yhat20, _ = K.average_forecast(X, pack["y_fit"], omegas20, blob["P_head"], blob["T"], [None])
_, strat20 = K.strategy_from_yhat(yhat20[0], y, KMZ_RETURN_STD, sigma)
s20 = span_ols_nw(strat20, y, S["kernel"]["strat"], mask_main)
s100 = span_ols_nw(S["rff"]["strat"], y, S["kernel"]["strat"], mask_main)
draw_tab = pd.DataFrame([
    {"draws": 20, "t_OLS": s20["t_ols"], "t_NW6": s20["t_nw"], "source": "rebuilt from cached X"},
    {"draws": 100, "t_OLS": s100["t_ols"], "t_NW6": s100["t_nw"], "source": "cache"},
    {"draws": "Nagel", "t_OLS": -0.12, "t_NW6": np.nan, "source": "Table I B"},
])
print(draw_tab.to_string(index=False, float_format=lambda v: f"{v:7.3f}"))
moved = abs(s100["t_ols"] - (-0.12)) < abs(s20["t_ols"] - (-0.12))
print(f"OLS t moves toward Nagel's −0.12 as draws go 20 → 100? {moved}  ({s20['t_ols']:.3f} → {s100['t_ols']:.3f})")



60 months give little power: even if the true IR were 0.28, the expected t-statistic is about 0.28 × sqrt(5) ≈ 0.6.

## 8. Homework — $T=60$ and $T=120$ (Nagel, Appendix C.1)

Guarded by `RUN_HOMEWORK`. Re-run kernel, vol-mom and a 5-draw ridgeless RFF at $P=12{,}000$. The classroom headline stays at $T=12$. Nagel's point is that the VoC / vol-mom story is a $T=12$ phenomenon.


In [ ]:
if not RUN_HOMEWORK:
    print("RUN_HOMEWORK=False: skipping T=60 and T=120. Flip the switch in the setup cell to run Nagel Appendix C.1.")
else:
    # 5 draws, not 100: homework, not the headline.
    rng_hw = np.random.default_rng(K.SEED + 99)
    omegas_hw = K.draw_omegas(rng_hw, 5, K.P_HEAD // 2, X.shape[1])
    hw_rows = []
    t_hw = time.time()
    for T_hw in (60, 120):
        yhat_r, _ = K.average_forecast(X, pack["y_fit"], omegas_hw, K.P_HEAD, T_hw, [None])
        yhat_k, n_r = K.rolling_kernel_forecast(X, pack["y_fit"], T_hw)
        yhat_v = K.volmom_forecast(X, y, T_hw)
        for name, yh, rstd in (("RFF", yhat_r[0], KMZ_RETURN_STD), ("kernel", yhat_k, KMZ_RETURN_STD), ("vol-mom", yhat_v, False)):
            sm = K.summarize_strategy(yh, y, sigma, rstd, ym)
            capm = sm["main"]["capm"]
            hw_rows.append({
                "T": T_hw,
                "strategy": name,
                "kernel_ridge": n_r if name == "kernel" else 0,
                "IR": capm["ir"],
                "t": capm["tstat"],
                "n": capm["n"],
            })
        print(f"T={T_hw} done  {time.time()-t_hw:.1f}s")
    print(pd.DataFrame(hw_rows).to_string(index=False, float_format=lambda v: f"{v:8.3f}"))
    print(f"homework {time.time()-t_hw:.1f}s")
